# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [4]:
# Load environment variables
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# Initialize LLM client
client = LLM(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY
)

# Load ChromaDB collection from Part 1
chroma_client = chromadb.PersistentClient(path="./chroma_db")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small"
)
collection = chroma_client.get_or_create_collection(
    name="games",
    embedding_function=embedding_fn
)

print(f"✓ ChromaDB collection loaded: {collection.count()} games available")

✓ ChromaDB collection loaded: 15 games available


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
from lib.tooling import tool

@tool(name="retrieve_game", description="Semantic search: Finds game results in the vector DB using ChromaDB. Returns the most relevant games matching the query with platform, name, year, and confidence score.")
def retrieve_game(query: str) -> dict:
    """
    Retrieves games from ChromaDB based on semantic similarity to the query.
    
    Args:
        query: A question about games or game-related topics
        
    Returns:
        dict with:
        - results: list of game documents with metadata
        - confidence: semantic search confidence score (1 - distance)
        - distance: raw distance metric from ChromaDB
    """
    # Retrieve from ChromaDB
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    
    # Format results
    retrieved_games = []
    if results['documents'] and len(results['documents']) > 0:
        for i, doc in enumerate(results['documents'][0]):
            distance = results['distances'][0][i] if results['distances'] else 0
            confidence = 1 - distance  # Convert distance to confidence
            
            metadata = results['metadatas'][0][i] if results['metadatas'] else {}
            
            retrieved_games.append({
                'document': doc,
                'platform': metadata.get('platform', 'Unknown'),
                'name': metadata.get('name', 'Unknown'),
                'year': metadata.get('year', 'Unknown'),
                'confidence': round(confidence, 2)
            })
    
    return {
        'results': retrieved_games,
        'query': query,
        'num_results': len(retrieved_games)
    }

#### Evaluate Retrieval Tool

In [6]:
from pydantic import BaseModel

class EvaluationReport(BaseModel):
    """Evaluation report for retrieved documents"""
    is_relevant: bool
    confidence: float
    reasoning: str

@tool(name="evaluate_retrieval", description="Evaluates if retrieved documents are relevant and sufficient to answer the user's question. Returns relevance judgment with confidence score and reasoning.")
def evaluate_retrieval(query: str, retrieval_result: dict) -> dict:
    """
    Uses an LLM to evaluate if retrieved documents are sufficient to answer the query.
    
    Args:
        query: The original user question
        retrieval_result: Results from retrieve_game tool
        
    Returns:
        dict with evaluation report including relevance, confidence, and reasoning
    """
    # Format documents for evaluation
    documents_text = "\n".join([
        f"- {r['name']} ({r['platform']}, {r['year']}): {r['document']}"
        for r in retrieval_result.get('results', [])
    ])
    
    # Create evaluation prompt
    eval_prompt = f"""Your task is to evaluate if the retrieved documents are sufficient to answer the user's question.

User Question: {query}

Retrieved Documents:
{documents_text}

Analyze whether these documents contain enough information to answer the question. Consider:
1. Do the documents directly address the question?
2. Is the information specific enough?
3. What is your confidence level (0.0 to 1.0)?

Provide a JSON response with:
- "is_relevant": boolean indicating if documents are useful
- "confidence": confidence score (0.0 to 1.0)
- "reasoning": explanation of your assessment
"""
    
    # Use LLM to evaluate (via the lib.llm.LLM wrapper created in Setup)
    response = client.invoke([UserMessage(content=eval_prompt)])
    eval_text = response.content
    
    # Try to parse JSON response
    try:
        import json
        # Extract JSON from response
        json_start = eval_text.find('{')
        json_end = eval_text.rfind('}') + 1
        if json_start != -1 and json_end > json_start:
            json_str = eval_text[json_start:json_end]
            eval_data = json.loads(json_str)
            report = EvaluationReport(
                is_relevant=eval_data.get('is_relevant', True),
                confidence=float(eval_data.get('confidence', 0.5)),
                reasoning=eval_data.get('reasoning', eval_text)
            )
        else:
            # Default if parsing fails
            report = EvaluationReport(
                is_relevant=True,
                confidence=0.7,
                reasoning=eval_text
            )
    except Exception as e:
        report = EvaluationReport(
            is_relevant=True,
            confidence=0.6,
            reasoning=f"Evaluation: {eval_text}"
        )
    
    return {
        'is_relevant': report.is_relevant,
        'confidence': report.confidence,
        'reasoning': report.reasoning
    }

#### Game Web Search Tool

In [7]:
from tavily import TavilyClient

@tool(name="game_web_search", description="Performs web search using Tavily API to find current information about games when internal knowledge is insufficient. Returns search results with answers and sources.")
def game_web_search(query: str) -> dict:
    """
    Searches the web for game-related information using Tavily API.
    
    Args:
        query: Search query about games
        
    Returns:
        dict with:
        - answer: synthesized answer from web search results
        - sources: list of source URLs
        - confidence: confidence in the answer (0.5-1.0 range)
    """
    try:
        # Initialize Tavily client
        tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
        
        # Perform web search
        search_response = tavily_client.search(query, max_results=5)
        
        # Extract answer and sources
        answer = search_response.get("answer", "No answer found")
        sources = [result.get("url", "") for result in search_response.get("results", [])]
        
        return {
            'answer': answer,
            'sources': sources,
            'confidence': 0.8,
            'query': query
        }
    except Exception as e:
        return {
            'answer': f"Web search failed: {str(e)}",
            'sources': [],
            'confidence': 0.0,
            'query': query,
            'error': str(e)
        }

### Agent

In [8]:
# Create Agent with system instructions and all tools
from lib.agents import Agent

# System prompt for the agent
system_prompt = """You are an expert gaming AI assistant specializing in video game information and history.

Your primary role is to:
1. Answer questions about video games using the game database (retrieve_game tool)
2. Evaluate if retrieved information is sufficient (evaluate_retrieval tool)
3. Fall back to web search for current or missing information (game_web_search tool)

Guidelines:
- First, try to retrieve information from the internal game database
- Evaluate the relevance of retrieved results
- If results are not relevant (confidence < 0.5) or insufficient, use web search
- Always provide sources and confidence levels in your responses
- Be accurate and cite game platforms, years, and publishers when available
- If you're unsure about information, acknowledge the uncertainty

When answering:
- Use retrieved data as primary source
- Cross-reference with web search for verification
- Provide complete answers with platform, release year, and relevant details
- Explain your reasoning for tool usage"""

# Create the agent
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=system_prompt,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3
)

print("✓ Agent created successfully with 3 tools: retrieve_game, evaluate_retrieval, game_web_search")

✓ Agent created successfully with 3 tools: retrieve_game, evaluate_retrieval, game_web_search


In [9]:
# Test queries with the agent
import json

test_queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?"
]

print("=" * 70)
print("AGENT TEST QUERIES")
print("=" * 70)

for i, query in enumerate(test_queries, 1):
    print(f"\n[Query {i}] {query}")
    print("-" * 70)
    
    try:
        # Invoke agent with session ID for state tracking
        response = agent.invoke(query, session_id=f"test_session_{i}")
        
        # Display response
        print(f"Answer: {response}")
        print()
    except Exception as e:
        print(f"Error: {str(e)}")
        print()

print("=" * 70)
print("✓ Agent testing complete")
print("=" * 70)

AGENT TEST QUERIES

[Query 1] When was Pokémon Gold and Silver released?
----------------------------------------------------------------------
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Error: 'str' object has no attribute 'get'


[Query 2] Which one was the first 3D platformer Mario game?
----------------------------------------------------------------------
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Error: 'str' object has no attribute 'get'


[Query 3] Was Mortal Kombat X released for Playstation 5?
----------------------------------------------------------------------
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep

### (Optional) Advanced

### (Optional) Advanced — Stateful agent + long-term memory

The cells below implement both optional TODOs:

1. **Convert the agent to a state machine, with the tools as pre-defined nodes.**
   `UdaPlayGraphAgent` builds an explicit `lib.state_machine` graph:

   ```
   entry → recall_memory → retrieve_game → evaluate_retrieval
                                               │
                        relevant & confident ──┴── not good enough
                                │                       │
                                ▼                       ▼
                             compose  ◄────────  game_web_search
                                │
                                ▼
                         persist_memory → end
   ```

   Every tool is a fixed node. The LLM only decides the branch (through
   `evaluate_retrieval`) and writes the final answer in `compose` — it never picks
   tools freely, so each run is deterministic and easy to trace.

2. **Update the agent with long-term memory.** A persistent, ChromaDB-backed
   `LongTermMemory` (from `lib.memory`) stores user facts and preferences as
   embeddings. `recall_memory` semantically retrieves relevant memories at the
   start of every run; `persist_memory` writes the interaction back at the end, so
   knowledge survives across separate `invoke()` calls and kernel restarts —
   unlike the per-conversation `ShortTermMemory` used by `lib.agents.Agent`.

> Requires a working `OPENAI_API_KEY` / `OPENAI_BASE_URL` (plus `TAVILY_API_KEY`
> for the web-search branch).

In [10]:
# ── (Optional) Advanced setup: long-term memory + a shared LLM ─────────────────
#
# lib.agents.Agent already runs on a state machine, but its ShortTermMemory only
# lasts for one conversation. Here we add a *long-term* store: user facts and
# preferences kept as embeddings in a persistent ChromaDB collection and searched
# semantically on every new query.

import os
import chromadb
from chromadb.utils import embedding_functions

from lib.llm import LLM
from lib.messages import SystemMessage, UserMessage
from lib.vector_db import VectorStoreManager
from lib.memory import LongTermMemory, MemoryFragment

# One LLM instance reused by every node of the graph.
llm = LLM(model="gpt-4o-mini", temperature=0.3)

# Identifies whose memories we read / write (a real app would use the logged-in user).
USER_ID = "student"


class PersistentVectorStoreManager(VectorStoreManager):
    """
    VectorStoreManager variant that:
      * persists to ./chroma_db (survives kernel restarts), and
      * points the OpenAI embedding function at OPENAI_BASE_URL with the same
        text-embedding-3-small model used in Part 1 (Vocareum-compatible).
    """

    def __init__(self, openai_api_key: str, path: str = "./chroma_db"):
        self.chroma_client = chromadb.PersistentClient(path=path)
        self.embedding_function = embedding_functions.OpenAIEmbeddingFunction(
            api_key=openai_api_key,
            model_name="text-embedding-3-small",
            api_base=os.getenv("OPENAI_BASE_URL"),
        )


class PersistentLongTermMemory(LongTermMemory):
    """LongTermMemory that reuses its collection instead of recreating it each run."""

    def __init__(self, db: VectorStoreManager, store_name: str = "long_term_memory"):
        self.vector_store = db.get_or_create_store(store_name)


memory_db = PersistentVectorStoreManager(os.getenv("OPENAI_API_KEY"))
ltm = PersistentLongTermMemory(memory_db)
print("✓ Long-term memory ready (ChromaDB collection: 'long_term_memory')")

✓ Long-term memory ready (ChromaDB collection: 'long_term_memory')


In [11]:
# ── (Optional) Advanced: the agent as an explicit state machine ─────────────────
#
# Instead of the generic "let the LLM call whatever tool it wants" loop in
# lib.agents.Agent, every tool becomes a fixed node with hard-wired transitions.
# The only dynamic decision is the branch after evaluate_retrieval.

from typing import Optional
from typing_extensions import TypedDict

from lib.state_machine import StateMachine, Step, EntryPoint, Termination


class GraphAgentState(TypedDict):
    user_query: str            # the question being answered
    owner: str                 # whose long-term memory to read / write
    relevance_threshold: float # min confidence to trust the internal DB
    recalled_memories: str     # memories pulled in by recall_memory
    retrieval: dict            # output of retrieve_game
    evaluation: dict           # output of evaluate_retrieval
    web_results: Optional[dict] # output of game_web_search (if used)
    used_web_search: bool       # which branch ran
    answer: str                 # final composed answer


class UdaPlayGraphAgent:
    """UdaPlay agent implemented as a lib.state_machine graph with one node per tool."""

    SYSTEM_PROMPT = (
        "You are UdaPlay, an expert video-game research assistant. "
        "Answer only from the context provided to you. Always state the platform, "
        "release year and publisher when they are available, note whether each fact "
        "came from the internal database or the web, and clearly say when something "
        "is unknown. Take the user's remembered preferences into account."
    )

    def __init__(self, ltm, llm, namespace: str = "udaplay",
                 default_owner: str = USER_ID, relevance_threshold: float = 0.5):
        self.ltm = ltm
        self.llm = llm
        self.namespace = namespace
        self.default_owner = default_owner
        self.relevance_threshold = relevance_threshold
        self.machine = self._build_machine()

    # ---- nodes ------------------------------------------------------------------
    def _recall_memory(self, state: GraphAgentState) -> dict:
        """Node: semantically recall what we already know about this user."""
        recalled = ""
        try:
            found = self.ltm.search(
                query_text=state["user_query"],
                owner=state["owner"],
                namespace=self.namespace,
                limit=3,
            )
            recalled = "\n".join(f"- {f.content}" for f in found.fragments)
        except Exception as exc:  # empty store, bad key, etc. -- keep going
            print(f"   [recall_memory] skipped ({exc})")
        print(f"   [recall_memory] {len(recalled.splitlines())} memory fragment(s)")
        return {"recalled_memories": recalled}

    def _retrieve_game(self, state: GraphAgentState) -> dict:
        """Node: the retrieve_game tool."""
        result = retrieve_game(query=state["user_query"])
        print(f"   [retrieve_game] {result['num_results']} candidate(s)")
        return {"retrieval": result}

    def _evaluate_retrieval(self, state: GraphAgentState) -> dict:
        """Node: the evaluate_retrieval tool."""
        report = evaluate_retrieval(
            query=state["user_query"], retrieval_result=state["retrieval"],
        )
        print(f"   [evaluate_retrieval] relevant={report['is_relevant']} "
              f"confidence={report['confidence']}")
        return {"evaluation": report}

    def _game_web_search(self, state: GraphAgentState) -> dict:
        """Node: the game_web_search tool (fallback branch)."""
        result = game_web_search(query=state["user_query"])
        print(f"   [game_web_search] {len(result.get('sources', []))} source(s)")
        return {"web_results": result, "used_web_search": True}

    def _compose(self, state: GraphAgentState) -> dict:
        """Node: write the final answer from whatever context the graph gathered."""
        blocks = []
        if state.get("recalled_memories"):
            blocks.append("Remembered about the user:\n" + state["recalled_memories"])
        docs = state["retrieval"].get("results", [])
        if docs:
            blocks.append("Internal game database:\n" +
                          "\n".join(f"- {d.get('document', '')}" for d in docs))
        if state.get("web_results"):
            web = state["web_results"]
            blocks.append(f"Web search result:\n{web.get('answer', '')}\n"
                          f"Sources: {', '.join(web.get('sources', [])) or 'none'}")
        context = "\n\n".join(blocks) if blocks else "No context available."

        answer = self.llm.invoke([
            SystemMessage(content=self.SYSTEM_PROMPT),
            UserMessage(content=f"Question: {state['user_query']}\n\nContext:\n{context}"),
        ]).content
        return {"answer": answer}

    def _persist_memory(self, state: GraphAgentState) -> dict:
        """Node: write this interaction back to long-term memory."""
        fact = (f"On being asked \"{state['user_query']}\", "
                f"the agent answered: {state['answer']}")
        try:
            self.ltm.register(MemoryFragment(
                content=fact[:600], owner=state["owner"], namespace=self.namespace,
            ))
            print("   [persist_memory] interaction stored")
        except Exception as exc:
            print(f"   [persist_memory] skipped ({exc})")
        return {}

    # ---- routing --------------------------------------------------------------
    def _route_after_eval(self, state: GraphAgentState) -> str:
        """Branch: trust the internal DB, or fall back to web search."""
        ev = state["evaluation"]
        good = ev.get("is_relevant") and \
            ev.get("confidence", 0.0) >= state["relevance_threshold"]
        return "compose" if good else "web_search"

    # ---- wiring -------------------------------------------------------------
    def _build_machine(self) -> StateMachine:
        machine = StateMachine[GraphAgentState](GraphAgentState)

        entry = EntryPoint[GraphAgentState]()
        recall = Step[GraphAgentState]("recall_memory", self._recall_memory)
        retrieve = Step[GraphAgentState]("retrieve_game", self._retrieve_game)
        evaluate = Step[GraphAgentState]("evaluate_retrieval", self._evaluate_retrieval)
        web_search = Step[GraphAgentState]("web_search", self._game_web_search)
        compose = Step[GraphAgentState]("compose", self._compose)
        persist = Step[GraphAgentState]("persist_memory", self._persist_memory)
        end = Termination[GraphAgentState]()

        machine.add_steps([entry, recall, retrieve, evaluate,
                           web_search, compose, persist, end])

        machine.connect(entry, recall)
        machine.connect(recall, retrieve)
        machine.connect(retrieve, evaluate)
        machine.connect(evaluate, [compose, web_search], self._route_after_eval)
        machine.connect(web_search, compose)
        machine.connect(compose, persist)
        machine.connect(persist, end)
        return machine

    # ---- public API -------------------------------------------------------
    def invoke(self, query: str, owner: Optional[str] = None) -> dict:
        """Run the graph for one question and return the final state dict."""
        initial: GraphAgentState = {
            "user_query": query,
            "owner": owner or self.default_owner,
            "relevance_threshold": self.relevance_threshold,
            "recalled_memories": "",
            "retrieval": {},
            "evaluation": {},
            "web_results": None,
            "used_web_search": False,
            "answer": "",
        }
        return self.machine.run(initial).get_final_state()


graph_agent = UdaPlayGraphAgent(ltm=ltm, llm=llm)
print("✓ UdaPlayGraphAgent built:",
      " -> ".join(["recall_memory", "retrieve_game", "evaluate_retrieval",
                   "[web_search]", "compose", "persist_memory"]))

✓ UdaPlayGraphAgent built: recall_memory -> retrieve_game -> evaluate_retrieval -> [web_search] -> compose -> persist_memory


In [12]:
# ── (Optional) Advanced: run the required project questions through the graph ───
project_questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for i, question in enumerate(project_questions, 1):
    print("\n" + "=" * 78)
    print(f"[{i}] {question}")
    print("=" * 78)
    try:
        final = graph_agent.invoke(question, owner=USER_ID)
        route = "web search" if final["used_web_search"] else "internal database"
        print(f"\nRoute taken : {route}")
        print(f"Answer      :\n{final['answer']}")
    except Exception as exc:
        print(f"Error: {exc}")


[1] When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
   [recall_memory] 2 memory fragment(s)
[StateMachine] Executing step: recall_memory
   [retrieve_game] 3 candidate(s)
[StateMachine] Executing step: retrieve_game
   [evaluate_retrieval] relevant=True confidence=0.9
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: compose
   [persist_memory] interaction stored
[StateMachine] Executing step: persist_memory
[StateMachine] Terminating: __termination__

Route taken : internal database
Answer      :
Pokémon Gold and Silver were released in 1999 for the Game Boy Color. This information comes from the internal database.

[2] Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
   [recall_memory] 3 memory fragment(s)
[StateMachine] Executing step: recall_memory
   [retrieve_game] 3 candidate(s)
[StateMachine] Executing step: retrieve_game
   [evaluate_retrieval] relevant=True confidence=0.9
[StateM

In [13]:
# ── (Optional) Advanced: long-term memory carried across separate invocations ──
#
# Long-term memory is per-user (the `owner` field) and independent of any session,
# so a fact learned in one `invoke()` is available in every later one.

# 1) Store a durable preference for this user.
ltm.register(MemoryFragment(
    content="The user collects Nintendo handheld consoles and prefers portable games.",
    owner=USER_ID,
    namespace=graph_agent.namespace,
))
print("Stored a user preference.\n")

# 2) Ask something where that preference should shape the answer. The recall_memory
#    node pulls the preference back in before `compose` writes the reply.
final = graph_agent.invoke(
    "From the internal database, which game would you recommend I add to my collection, and why?",
    owner=USER_ID,
)
print("\n" + "-" * 78)
print(final["answer"])

# 3) Everything the agent has learned about this user so far.
print("\n--- Long-term memory for", USER_ID, "---")
recall = ltm.search(
    query_text=" ", owner=USER_ID, namespace=graph_agent.namespace, limit=20,
)
for frag in recall.fragments:
    print("•", frag.content)

Stored a user preference.

[StateMachine] Starting: __entry__
   [recall_memory] 3 memory fragment(s)
[StateMachine] Executing step: recall_memory
   [retrieve_game] 3 candidate(s)
[StateMachine] Executing step: retrieve_game
   [evaluate_retrieval] relevant=False confidence=0.2
[StateMachine] Executing step: evaluate_retrieval
   [game_web_search] 5 source(s)
[StateMachine] Executing step: web_search
[StateMachine] Executing step: compose
   [persist_memory] interaction stored
[StateMachine] Executing step: persist_memory
[StateMachine] Terminating: __termination__

------------------------------------------------------------------------------
Based on your preferences for Nintendo handheld consoles and portable games, I recommend adding **Pokémon Ruby and Sapphire** to your collection. 

- **Platform**: Game Boy Advance
- **Release Year**: 2002
- **Publisher**: Nintendo

These games are part of the third generation of Pokémon and are set in the Hoenn region, featuring new Pokémon and